Voici le notebook d'application de la théorie des valeurs extrêmes sur le jeu de données NBA.

## 1 Import/Initialisation 

Je reprends ici principalement le code du nba_api.ipynb afin de définir les différentes fonctions d'import etc

In [2]:
!pip install nba_api


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import time

from nba_api.stats.endpoints import (
    leaguegamefinder,
    boxscoretraditionalv2,
    playbyplayv3,
    teamgamelog,
)
from nba_api.stats.static import teams, players

from utils import nba_request_with_retry, get_team_id  ,get_box, get_extended_pbp, get_pbp, get_games_played, point_filter1, fetch_pbp_for_games, calculate_weighted_score, SLEEP




### Données anecdotiques

In [4]:
# Toutes les équipes NBA
all_teams = teams.get_teams()
df_teams = pd.DataFrame(all_teams)
print(f"Nombre d'équipes : {len(df_teams)}")
df_teams.head(10)

Nombre d'équipes : 30


,id,full_name,abbreviation,nickname,city,state,year_founded
0,1610612737,Atlanta Hawks,ATL,Hawks,Atlanta,Georgia,1949
1,1610612738,Boston Celtics,BOS,Celtics,Boston,Massachusetts,1946
2,1610612739,Cleveland Cavaliers,CLE,Cavaliers,Cleveland,Ohio,1970
3,1610612740,New Orleans Pelicans,NOP,Pelicans,New Orleans,Louisiana,2002
4,1610612741,Chicago Bulls,CHI,Bulls,Chicago,Illinois,1966
5,1610612742,Dallas Mavericks,DAL,Mavericks,Dallas,Texas,1980
6,1610612743,Denver Nuggets,DEN,Nuggets,Denver,Colorado,1976
7,1610612744,Golden State Warriors,GSW,Warriors,San Francisco,California,1946
8,1610612745,Houston Rockets,HOU,Rockets,Houston,Texas,1967
9,1610612746,Los Angeles Clippers,LAC,Clippers,Los Angeles,California,1970


In [5]:
gsw = get_team_id("Golden State Warriors")
bos = get_team_id("Boston Celtics")
print("Golden State Warriors :", gsw)
print("Boston Celtics        :", bos)

Golden State Warriors : {'id': 1610612744, 'full_name': 'Golden State Warriors', 'abbreviation': 'GSW', 'nickname': 'Warriors', 'city': 'San Francisco', 'state': 'California', 'year_founded': 1946}
Boston Celtics        : {'id': 1610612738, 'full_name': 'Boston Celtics', 'abbreviation': 'BOS', 'nickname': 'Celtics', 'city': 'Boston', 'state': 'Massachusetts', 'year_founded': 1946}


In [ ]:
SEASON = "2023-24"
TEAM_ID = gsw["id"]  # Golden State Warriors

df_games = get_games_played(TEAM_ID, SEASON)
print(f"Matchs récupérés : {len(df_games)}")
df_games[["Game_ID", "GAME_DATE", "MATCHUP", "WL", "PTS"]].head(10)

Tentative 1/4 échouée (ReadTimeout). Nouvelle tentative dans 3s…


In [ ]:
# Sélectionner un match précis pour la suite (1er match de la liste)
GAME_ID = df_games["Game_ID"].iloc[0]
print(f"Match sélectionné : {df_games['MATCHUP'].iloc[0]}  —  {df_games['GAME_DATE'].iloc[0]}")
print(f"Game ID           : {GAME_ID}")

In [ ]:
box = get_box(game_id = GAME_ID)

df_players_box = box.get_data_frames()[0]
df_teams_box   = box.get_data_frames()[1]

cols_joueur = ["TEAM_ABBREVIATION", "PLAYER_NAME", "MIN", "PTS", "REB", "AST", "STL", "BLK", "TO", "PLUS_MINUS"]
print("=== Box Score Joueurs ===")
df_players_box[cols_joueur].dropna(subset=["MIN"])


In [ ]:
print("=== Box Score Équipes ===")
df_teams_box[["TEAM_ABBREVIATION", "PTS", "FG_PCT", "FG3_PCT", "FT_PCT", "REB", "AST", "STL", "BLK", "TO"]]

In [ ]:
df_pbp = get_pbp(game_id = GAME_ID)
df_pbp.head(10)

In [ ]:
df = get_extended_pbp(game_id = GAME_ID)

print("Aperçu enrichi :")
df[["ELAPSED_MINUTES", "period", "clock", "description",
    "SCORE_HOME", "SCORE_VISITOR", "SCOREMARGIN_FF"]].dropna(
    subset=["ELAPSED_MINUTES"]
).head(15)

In [ ]:
# Récupérer les noms des deux équipes depuis le box score
home_team = df_teams_box.iloc[0]["TEAM_ABBREVIATION"]
away_team = df_teams_box.iloc[1]["TEAM_ABBREVIATION"]

In [ ]:
runs = point_filter1(game_id = GAME_ID)

# Top 10 des plus grands runs
top_runs = runs.sort_values("pts", ascending=False).head(10)
print("Top 10 des runs du match :")
top_runs[["SCORER", "pts", "start_min", "end_min", "n_events"]].rename(
    columns={"SCORER":"Équipe","pts":"Points marqués","start_min":"Début (min)",
             "end_min":"Fin (min)","n_events":"Nb actions"}
).round(2)

In [ ]:
N_GAMES = 10
game_ids_sample = df_games["Game_ID"].iloc[:N_GAMES].tolist()
print(f"Récupération du PBP de {N_GAMES} matchs des GSW ({SEASON})…")
df_all_pbp = fetch_pbp_for_games(game_ids_sample)
print(f" {len(df_all_pbp)} actions collectées sur {N_GAMES} matchs.")

In [ ]:
# Score différentiel moyen par tranche de 1 minute (depuis la perspective de GSW)
# On récupère si GSW était HOME ou VISITOR pour chaque match
def get_home_flag(gid, team_id):
    """Renvoie +1 si team_id était HOME, -1 si VISITOR (signe de SCOREMARGIN)."""
    row = df_games[df_games["Game_ID"] == gid]
    if row.empty:
        return 1
    matchup = row["MATCHUP"].values[0]
    # "GSW vs. XXX" → home  /  "GSW @ XXX" → visitor
    return 1 if "vs." in matchup else -1

df_all_pbp["HOME_FLAG"] = df_all_pbp["GAME_ID"].apply(lambda gid: get_home_flag(gid, TEAM_ID))
# Marge vue par GSW
df_all_pbp["MARGIN_GSW"] = df_all_pbp["SCOREMARGIN_FF"] * df_all_pbp["HOME_FLAG"]

# Arrondir au minute inférieure
df_all_pbp["MINUTE_BIN"] = df_all_pbp["ELAPSED_MINUTES"].apply(
    lambda x: int(x) if pd.notna(x) else np.nan
)

agg = (
    df_all_pbp[df_all_pbp["MINUTE_BIN"].between(0, 47)]
    .groupby("MINUTE_BIN")["MARGIN_GSW"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

## 2 Réponses aux questions

Q1 — Quelle est la probabilité d'observer un run extrême ?
       (niveau brut ou niveau score pondéré)

In [ ]:
import numpy as np

# Define what an 'extreme run' means. For example, runs with >= 8 points.
extreme_threshold = 8

# Filter runs that meet the extreme threshold
extreme_runs = runs[runs['pts'] >= extreme_threshold]

# Calculate the total number of runs observed
total_runs_count = len(runs)

# Calculate the number of extreme runs
extreme_runs_count = len(extreme_runs)

# Calculate the probability of an extreme run
probability_extreme_run = extreme_runs_count / total_runs_count if total_runs_count > 0 else 0

print(f"Total runs analyzed: {total_runs_count}")
print(f"Number of extreme runs (>= {extreme_threshold} points): {extreme_runs_count}")
print(f"Probability of observing an extreme run: {probability_extreme_run:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.histplot(runs['pts'], bins=20, kde=True)
plt.axvline(x=extreme_threshold, color='red', linestyle='--', label=f'Extreme Run Threshold ({extreme_threshold} pts)')
plt.title('Distribution of Run Points')
plt.xlabel('Points Scored in a Run')
plt.ylabel('Frequency')
plt.legend()
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
df_all_pbp[['WEIGHTED_SCORE', 'WEIGHTED_SCORER']] = df_all_pbp.apply(calculate_weighted_score, axis=1, result_type='expand')

print("First 10 rows of df_all_pbp with new weighted score columns:")
print(df_all_pbp[['actionType', 'shotValue', 'description', 'location', 'WEIGHTED_SCORE', 'WEIGHTED_SCORER']].head(10))

In [ ]:
df_weighted_score_events = df_all_pbp[df_all_pbp['WEIGHTED_SCORE'] > 0].copy()

# Identify who scores: 'HOME' or 'VISITOR' based on WEIGHTED_SCORER
df_weighted_score_events["SCORER"] = df_weighted_score_events["WEIGHTED_SCORER"]

# Group events by game and then identify consecutive weighted scoring by the same team
df_weighted_score_events = df_weighted_score_events.sort_values(by=['GAME_ID', 'ELAPSED_SECONDS']).reset_index(drop=True)
df_weighted_score_events["WEIGHTED_RUN_ID"] = (
    df_weighted_score_events.groupby('GAME_ID')['SCORER'].bfill() !=
    df_weighted_score_events.groupby('GAME_ID')['SCORER'].bfill().shift()
).cumsum()

# Aggregate by weighted run
weighted_runs = df_weighted_score_events.groupby(['GAME_ID', 'WEIGHTED_RUN_ID', 'SCORER']).agg(
    weighted_pts=("WEIGHTED_SCORE", "sum"),
    start_min=("ELAPSED_MINUTES", "first"),
    end_min=("ELAPSED_MINUTES", "last"),
    n_events=("actionType", "count"),
).reset_index()

# Top 10 of the largest weighted runs
top_weighted_runs = weighted_runs.sort_values("weighted_pts", ascending=False).head(10)
print("Top 10 weighted runs of the match:")
print(top_weighted_runs[["SCORER", "weighted_pts", "start_min", "end_min", "n_events"]].rename(
    columns={
        "SCORER": "Équipe",
        "weighted_pts": "Points pondérés",
        "start_min": "Début (min)",
        "end_min": "Fin (min)",
        "n_events": "Nb actions"
    })
.round(2))


In [ ]:
import numpy as np

# Define what an 'extreme weighted run' means. For example, runs with >= 10 weighted points.
extreme_weighted_threshold = 10

# Filter runs that meet the extreme threshold
extreme_weighted_runs = weighted_runs[weighted_runs['weighted_pts'] >= extreme_weighted_threshold]

# Calculate the total number of weighted runs observed
total_weighted_runs_count = len(weighted_runs)

# Calculate the number of extreme weighted runs
extreme_weighted_runs_count = len(extreme_weighted_runs)

# Calculate the probability of an extreme weighted run
probability_extreme_weighted_run = extreme_weighted_runs_count / total_weighted_runs_count if total_weighted_runs_count > 0 else 0

print(f"Total weighted runs analyzed: {total_weighted_runs_count}")
print(f"Number of extreme weighted runs (>= {extreme_weighted_threshold} weighted points): {extreme_weighted_runs_count}")
print(f"Probability of observing an extreme weighted run: {probability_extreme_weighted_run:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(weighted_runs['weighted_pts'], bins=20, kde=True)
plt.axvline(x=extreme_weighted_threshold, color='red', linestyle='--', label=f'Extreme Weighted Run Threshold ({extreme_weighted_threshold} pts)')
plt.title('Distribution of Weighted Run Points')
plt.xlabel('Weighted Points Scored in a Run')
plt.ylabel('Frequency')
plt.legend()
plt.grid(axis='y', alpha=0.75)
plt.show()